# 04 · The training loop: AdamW, schedules, clipping, checkpoints

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/04_training_loop.ipynb) &nbsp; 📖 Concept deck: [The training loop](https://nilesh-salpe.github.io/visual-ai/pytorch/04-training-loop/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** a standard step (forward, loss, `zero_grad`, backward, clip, optimizer and scheduler step), parameter groups that exclude norms and biases from weight decay, a warmup + cosine `LambdaLR`, and checkpoints that resume bit-for-bit.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

def toy_model(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(8, 32), nn.LayerNorm(32), nn.GELU(), nn.Linear(32, 1))
gX = torch.Generator().manual_seed(1); DATA_X = torch.randn(256, 8, generator=gX); DATA_Y = (DATA_X[:, :1] * 2 - DATA_X[:, 1:2]).sin()

## Task 1: Weight decay only on matrices

LLM recipes apply weight decay to 2-D weight matrices but not to biases or norm gains. Return two parameter groups for AdamW: `[{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}]`, splitting by `p.dim() >= 2`.

In [ ]:
def param_groups(model, wd=0.1):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_param_groups(f):
    m = toy_model(); g = f(m, 0.1)
    assert len(g) == 2 and g[0]["weight_decay"] == 0.1 and g[1]["weight_decay"] == 0.0
    assert sum(p.numel() for p in g[0]["params"]) == 8 * 32 + 32 * 1, "only the two Linear weight matrices decay"
    assert len(g[1]["params"]) == 4, "two biases + LayerNorm weight and bias"
    torch.optim.AdamW(g, lr=1e-3)

check("param_groups", _test_param_groups, param_groups)

## Task 2: Warmup + cosine with LambdaLR

`torch.optim.lr_scheduler.LambdaLR(opt, fn)` multiplies the base learning rate by `fn(step)`. Write `make_scheduler(opt, warmup, total, final=0.1)` with linear warmup to 1 over `warmup` steps, then cosine decay to `final` at `total`.

In [ ]:
def make_scheduler(opt, warmup, total, final=0.1):
    def fn(step):
        # TODO
        raise NotImplementedError
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_make_scheduler(f):
    opt = torch.optim.AdamW(toy_model().parameters(), lr=1.0)
    s = f(opt, 10, 110); lrs = []
    for _ in range(111):
        lrs.append(opt.param_groups[0]["lr"]); opt.step(); s.step()
    assert abs(lrs[0] - 0.1) < 1e-9 and abs(lrs[9] - 1.0) < 1e-9, "warmup: (step+1)/warmup"
    assert abs(lrs[60] - 0.55) < 1e-9 and abs(lrs[110] - 0.1) < 1e-9, "cosine down to 10%"

check("make_scheduler", _test_make_scheduler, make_scheduler)

## Task 3: Train, checkpoint, resume

Write `train(model, opt, sched, steps, start=0)` that runs full-batch steps on `DATA_X, DATA_Y`: `opt.zero_grad(set_to_none=True)`, MSE loss, backward, `clip_grad_norm_(…, 1.0)`, `opt.step()`, `sched.step()`. Return the list of losses.
The test trains 40 steps straight, and separately 20 steps → save `{"model", "opt", "sched"}` state dicts → load into fresh objects → 20 more. Both runs must end with identical weights.

In [ ]:
def train(model, opt, sched, steps):
    losses = []
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_train(f):
    import io
    pg = globals().get("param_groups_solution") or param_groups; ms = globals().get("make_scheduler_solution") or make_scheduler
    def fresh():
        m = toy_model(); o = torch.optim.AdamW(pg(m), lr=3e-2); return m, o, ms(o, 5, 40)
    m1, o1, s1 = fresh(); l1 = f(m1, o1, s1, 40)
    assert l1[-1] < 0.5 * l1[0], "loss should fall"
    m2, o2, s2 = fresh(); f(m2, o2, s2, 20)
    buf = io.BytesIO(); torch.save({"model": m2.state_dict(), "opt": o2.state_dict(), "sched": s2.state_dict()}, buf); buf.seek(0)
    m3, o3, s3 = fresh(); ck = torch.load(buf)
    m3.load_state_dict(ck["model"]); o3.load_state_dict(ck["opt"]); s3.load_state_dict(ck["sched"])
    f(m3, o3, s3, 20)
    for a, b in zip(m1.parameters(), m3.parameters()):
        assert torch.equal(a, b), "resuming from a checkpoint must reproduce the uninterrupted run exactly"

check("train", _test_train, train)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Weight decay only on matrices
def param_groups_solution(model, wd=0.1):
    decay = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]
    return [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}]

check("param_groups (reference)", _test_param_groups, param_groups_solution)

In [ ]:
# @title Solution for Task 2: Warmup + cosine with LambdaLR
import math
def make_scheduler_solution(opt, warmup, total, final=0.1):
    def fn(step):
        if step < warmup:
            return (step + 1) / warmup
        p = min(1.0, (step - warmup) / max(1, total - warmup))
        return final + (1 - final) * 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

check("make_scheduler (reference)", _test_make_scheduler, make_scheduler_solution)

In [ ]:
# @title Solution for Task 3: Train, checkpoint, resume
def train_solution(model, opt, sched, steps):
    losses = []
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = F.mse_loss(model(DATA_X), DATA_Y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); losses.append(loss.item())
    return losses

check("train (reference)", _test_train, train_solution)

---
## Experiments
**What a checkpoint must contain.** Saving only the model weights and resuming changes the trajectory: AdamW's moment estimates and the scheduler position are lost.

In [ ]:
import io
def fresh():
    m = toy_model(); o = torch.optim.AdamW(param_groups_solution(m), lr=3e-2); return m, o, make_scheduler_solution(o, 5, 40)
m0, o0, s0 = fresh(); train_solution(m0, o0, s0, 40)
def run(save_opt):
    m, o, s = fresh(); train_solution(m, o, s, 20)
    ck = {"model": m.state_dict()} | ({"opt": o.state_dict(), "sched": s.state_dict()} if save_opt else {})
    m2, o2, s2 = fresh(); m2.load_state_dict(ck["model"])
    if save_opt: o2.load_state_dict(ck["opt"]); s2.load_state_dict(ck["sched"])
    train_solution(m2, o2, s2, 20)
    return max((a - b).abs().max().item() for a, b in zip(m0.parameters(), m2.parameters()))
print(f"max weight difference vs the uninterrupted run, resuming with optimizer + scheduler state: {run(True):.2e}")
print(f"max weight difference vs the uninterrupted run, resuming with weights only:             {run(False):.2e}")
print("(weights only restarts warmup and Adam's moments, so the run takes a different path)")
print("AdamW defaults:", {k: v for k, v in torch.optim.AdamW([torch.zeros(1, requires_grad=True)]).defaults.items() if k in ("lr", "betas", "eps", "weight_decay")})

---
## Takeaways
* One step: `zero_grad` → forward → loss → `backward` → clip → `optimizer.step()` → `scheduler.step()`.
* Exclude biases and norm gains from weight decay with parameter groups; AdamW's PyTorch default weight decay is 0.01, LLM recipes often use 0.1.
* A resumable checkpoint stores model, optimizer and scheduler state (and RNG/data position in real runs).